# Model Training

Train ML models for price prediction.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

print("="*70)
print("MODEL TRAINING ON KAGGLE DATA")
print("="*70)
print(f"Training Period: {TRAIN_START} to {TRAIN_END}")
print(f"Test Period: {TEST_START} to {TEST_END}")
print("="*70)

✓ Imports loaded successfully


In [ ]:
# Load and prepare data for first ticker
ticker = DEFAULT_TICKERS[0]
print(f"\nLoading {ticker}...")

raw_data = load_kaggle_data(ticker)
cleaned_data = clean_ohlcv_data(raw_data)

print(f"Cleaned data shape: {cleaned_data.shape}")
print(f"Date range: {cleaned_data.index[0]} to {cleaned_data.index[-1]}")

# Split by dates
train_data, test_data = split_data_by_date(cleaned_data)
print(f"\nTrain data: {train_data.shape}")
print(f"Test data: {test_data.shape}")

Available raw data intervals:
--------------------------------------------------

RELIANCE.NS:
  ✓ 1d
  ✓ 5m
  ✓ 1m

TCS.NS:
  ✓ 1d
  ✓ 5m
  ✓ 1m

INFY.NS:
  ✓ 1d
  ✓ 5m
  ✓ 1m

HDFCBANK.NS:
  ✓ 1d
  ✓ 5m
  ✓ 1m

ICICIBANK.NS:
  ✓ 1d
  ✓ 5m
  ✓ 1m


## Check Available Data Intervals

In [3]:
%pip install seaborn --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Add features
def add_basic_features(data):
    df = data.copy()
    
    # Returns
    df['returns'] = df['Close'].pct_change()
    df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # Moving averages
    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['SMA_50'] = df['Close'].rolling(50).mean()
    
    # Price range
    df['range'] = df['High'] - df['Low']
    df['range_pct'] = df['range'] / df['Close']
    
    # Volume features
    df['Volume_SMA_20'] = df['Volume'].rolling(20).mean()
    df['Volume_ratio'] = df['Volume'] / df['Volume_SMA_20']
    
    # Volatility
    df['volatility_20'] = df['returns'].rolling(20).std()
    
    # RSI
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Target: 1 if next close > current close, else 0
    df['target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    
    return df.dropna()

# Add features
train_with_features = add_basic_features(train_data)
test_with_features = add_basic_features(test_data)

print(f"\nTrain with features: {train_with_features.shape}")
print(f"Test with features: {test_with_features.shape}")
print(f"Features: {[col for col in train_with_features.columns if col != 'target']}")

Loaded features from: ..\data\indicators\TCS.NS_features.csv
2025-12-19 18:47:11 - src.modeling.train_model - INFO - Prepared data | Train: (862, 39) | Test: (216, 39)
Training samples: 862
Test samples: 216
Features: 39


In [ ]:
# Prepare X and y
feature_cols = [col for col in train_with_features.columns if col not in ['target', 'Open', 'High', 'Low', 'Close', 'Volume']]

X_train = train_with_features[feature_cols]
y_train = train_with_features['target']

X_test = test_with_features[feature_cols]
y_test = test_with_features['target']

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Target distribution (train): {y_train.value_counts().to_dict()}")

2025-12-19 18:47:11 - src.modeling.train_model - INFO - Training stacked model (RandomForest + XGBoost)...
2025-12-19 18:47:17 - src.modeling.train_model - INFO - Stacked model training completed


In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled train mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"Scaled test mean: {X_test_scaled.mean():.4f}, std: {X_test_scaled.std():.4f}")

2025-12-19 18:47:17 - src.modeling.evaluate_model - INFO - 
Stacked (RF + XGBoost) Performance:
2025-12-19 18:47:17 - src.modeling.evaluate_model - INFO - Accuracy: 0.4954
2025-12-19 18:47:17 - src.modeling.evaluate_model - INFO - Precision: 0.5024
2025-12-19 18:47:17 - src.modeling.evaluate_model - INFO - Recall: 0.9455
2025-12-19 18:47:17 - src.modeling.evaluate_model - INFO - F1 Score: 0.6562
2025-12-19 18:47:17 - src.modeling.evaluate_model - INFO - 
Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.03      0.05       106
           1       0.50      0.95      0.66       110

    accuracy                           0.50       216
   macro avg       0.42      0.49      0.35       216
weighted avg       0.42      0.50      0.36       216



In [ ]:
# Train Random Forest
print("\nTraining RandomForest classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_train = rf_model.predict(X_train_scaled)
y_pred_test = rf_model.predict(X_test_scaled)

train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

2025-12-19 18:47:17 - src.modeling.train_model - INFO - Saved model artifacts for TCS.NS
Model saved successfully!


In [ ]:
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_test))

Stacked Model Components:
Base Learners: [RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42), XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=1,
              num_parallel_tree=None, ...)]
Final Estimator: LogisticRegression(max_iter=1000)


## Improve Model Accuracy

Try these optimization strategies to increase accuracy:

In [ ]:
# Train models for all tickers
results = {}

print("\n" + "="*70)
print("TRAINING ALL TICKERS")
print("="*70)

for ticker in DEFAULT_TICKERS:
    print(f"\nTraining {ticker}...", end=" ")
    
    try:
        # Load and prepare
        raw_data = load_kaggle_data(ticker)
        cleaned = clean_ohlcv_data(raw_data)
        train, test = split_data_by_date(cleaned)
        
        # Features
        train_f = add_basic_features(train)
        test_f = add_basic_features(test)
        
        if len(train_f) == 0 or len(test_f) == 0:
            print("SKIPPED (no data)")
            continue
        
        # Train model
        X_train = train_f[feature_cols]
        y_train = train_f['target']
        X_test = test_f[feature_cols]
        y_test = test_f['target']
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
        model.fit(X_train_scaled, y_train)
        
        test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
        
        results[ticker] = {
            'model': model,
            'scaler': scaler,
            'features': feature_cols,
            'train_samples': len(X_train),
            'test_samples': len(X_test),
            'test_accuracy': test_acc
        }
        
        print(f"✅ Accuracy: {test_acc:.4f} | Train: {len(X_train)} | Test: {len(X_test)}")
        
    except Exception as e:
        print(f"❌ Error: {str(e)[:50]}")

print("\n" + "="*70)
print(f"Successfully trained {len(results)} models")
print("="*70)

2025-12-19 18:47:17 - src.modeling.train_model - INFO - Training improved stacked model (RF + XGB + GB)...
2025-12-19 18:47:26 - src.modeling.train_model - INFO - Improved stacked model training completed
Original Stacked Model Accuracy: 0.4954
Improved Stacked Model Accuracy: 0.5000
Improvement: 0.46%
2025-12-19 18:47:26 - src.modeling.evaluate_model - INFO - 
Improved Stacked (RF + XGB + GB) Performance:
2025-12-19 18:47:26 - src.modeling.evaluate_model - INFO - Accuracy: 0.5000
2025-12-19 18:47:26 - src.modeling.evaluate_model - INFO - Precision: 0.5047
2025-12-19 18:47:26 - src.modeling.evaluate_model - INFO - Recall: 0.9727
2025-12-19 18:47:26 - src.modeling.evaluate_model - INFO - F1 Score: 0.6646
2025-12-19 18:47:26 - src.modeling.evaluate_model - INFO - 
Classification Report:
              precision    recall  f1-score   support

           0       0.25      0.01      0.02       106
           1       0.50      0.97      0.66       110

    accuracy                           0

In [10]:
# Option 2: Use GridSearchCV to find best XGBoost hyperparameters
best_xgb, best_cv_score = tune_xgboost_hyperparameters(X_train, y_train)

# Then train stacked model with tuned hyperparameters
model_tuned, tuned_accuracy = train_tuned_stacked_model(X_train, y_train, X_test, y_test)
y_pred_tuned = model_tuned.predict(X_test)

print(f"\n--- Accuracy Comparison ---")
print(f"Original Model:  {acc_original:.4f}")
print(f"Improved Model:  {acc_improved:.4f}")
print(f"Tuned Model:     {tuned_accuracy:.4f}")

# Evaluate tuned model
metrics_tuned = evaluate_classification(y_test, y_pred_tuned, model_name='Tuned Stacked (RF + XGB + GB)')

2025-12-19 18:47:26 - src.modeling.train_model - INFO - Starting XGBoost hyperparameter tuning...
Fitting 3 folds for each of 81 candidates, totalling 243 fits
2025-12-19 18:47:43 - src.modeling.train_model - INFO - Best XGBoost parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
2025-12-19 18:47:43 - src.modeling.train_model - INFO - Best CV score: 0.5255
2025-12-19 18:47:43 - src.modeling.train_model - INFO - Training stacked model with tuned hyperparameters...
2025-12-19 18:47:43 - src.modeling.train_model - INFO - Starting XGBoost hyperparameter tuning...
Fitting 3 folds for each of 81 candidates, totalling 243 fits
2025-12-19 18:47:59 - src.modeling.train_model - INFO - Best XGBoost parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
2025-12-19 18:47:59 - src.modeling.train_model - INFO - Best CV score: 0.5255
2025-12-19 18:48:03 - src.modeling.train_model - INFO - Tuned stacked model training completed
20

### Other Strategies to Improve Accuracy:

1. **Feature Engineering**: Create new features or remove irrelevant ones
2. **Data Preprocessing**: Better handling of outliers, normalization
3. **Class Imbalance**: Use SMOTE or adjust class weights further
4. **Ensemble**: Add more diverse base learners (SVM, Neural Networks)
5. **Target Engineering**: Ensure target variable is well-defined
6. **Feature Scaling**: Try different scaling methods